# Identifying and tracking paranoia require distinct behavioral signatures

## Fully reproducible manuscript analyses

This notebook runs from pseudonymized trial-level choices and weekly measures to the manuscript’s main Figures 2–4, Supplementary Figures 1–3 and Supplementary Tables 1–2. Run all cells from top to bottom from the repository root.

In [1]:
from __future__ import annotations

from pathlib import Path
import platform
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import statsmodels
import statsmodels.formula.api as smf
from IPython.display import display

warnings.filterwarnings("ignore", category=RuntimeWarning)

# The notebook works whether it is launched from the repository root or a child folder.
start = Path.cwd().resolve()
ROOT = next((p for p in [start, *start.parents] if (p / "data").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate the repository root containing data/.")

DATA = ROOT / "data"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

SEED = 20260724
N_PEOPLE = 150
N_PERSON_WEEKS = 1_000
N_TRIALS = 160
PARANOIA_THRESHOLD = 11
ROLLING_WINDOW = 9
MIN_REWARDED_PER_WINDOW = 3
MIN_VALID_WINDOWS = 20
FACETS = ["switch_rate", "switch_momentum", "structure", "reward_sensitivity"]

COLORS = {
    "symptom": "#A87884",
    "wsr": "#8A959C",
    "rigidity": "#C4A06B",
    "trait": "#2F6F6A",
    "state": "#C88A1E",
    "volatility": "#6B5B95",
    "blue": "#718A99",
    "dark": "#333333",
    "light": "#D8D8D8",
}

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 1.0,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "svg.fonttype": "none",
})

print(
    f"Python {platform.python_version()} | pandas {pd.__version__} | "
    f"NumPy {np.__version__} | SciPy {scipy.__version__} | "
    f"statsmodels {statsmodels.__version__}"
)

Python 3.11.9 | pandas 2.2.3 | NumPy 2.1.3 | SciPy 1.15.1 | statsmodels 0.14.4


In [2]:
# Public, pseudonymized inputs. This is the only data-loading cell.
required_files = {
    "trials": "prl_trials.csv.gz",
    "weekly": "weekly_measures.csv",
    "demographics": "supplementary_table_1_sample_characteristics.csv",
    "retention": "supplementary_table_2_weekly_retention.csv",
    "analysis_samples": "supplementary_table_2_analysis_samples.csv",
}
missing = [name for name in required_files.values() if not (DATA / name).exists()]
if missing:
    raise FileNotFoundError("Missing public analysis files: " + ", ".join(missing))

trials = pd.read_csv(DATA / required_files["trials"])
weekly = pd.read_csv(DATA / required_files["weekly"])
demographics_source = pd.read_csv(DATA / required_files["demographics"])
retention_source = pd.read_csv(DATA / required_files["retention"])
analysis_samples_source = pd.read_csv(DATA / required_files["analysis_samples"])

assert set(trials.columns) == {"participant_id", "week", "trial", "choice", "rewarded"}
assert len(trials) == N_PERSON_WEEKS * N_TRIALS
assert trials.groupby(["participant_id", "week"]).size().eq(N_TRIALS).all()
assert len(weekly) == N_PERSON_WEEKS
assert weekly["participant_id"].nunique() == N_PEOPLE
assert not any("worker" in column.lower() for column in [*trials.columns, *weekly.columns])

print(f"Loaded {len(trials):,} trials from {N_PERSON_WEEKS:,} sessions.")
print(f"Loaded weekly measures for {weekly['participant_id'].nunique()} participants.")

Loaded 160,000 trials from 1,000 sessions.
Loaded weekly measures for 150 participants.


## 1. Preprocessing and canonical person-week panel

- The primary panel contains **150 participants and 1,000 paired person-weeks**.
- Weekly persecution is the sum of ten R-GPTS persecution items, each scored 0–4; all ten items are required.
- Win-switch rate is the proportion of rewarded choices followed by a deck change.
- Win-switch rigidity is the sign-inverted equal-weight mean of standardized switchiness, switch momentum, rolling-nine temporal structure and reward sensitivity.
- Structure is the sample standard deviation of the nine-trial rolling win-switch rate, requiring at least three rewarded transitions per window and 20 valid windows per session.
- `mu03` is the supplied session-level HGF volatility-prior estimate. This notebook analyses that fitted parameter; it does not refit the HGF.

Participant pseudonyms preserve repeated-measures structure. No recruitment-platform identifiers or individual demographic records are loaded.

In [3]:
def zscore_sample(series: pd.Series) -> pd.Series:
    """Standardize using the sample standard deviation (ddof=1)."""
    values = pd.to_numeric(series, errors="coerce")
    sd = values.std(ddof=1)
    if not np.isfinite(sd) or sd == 0:
        return pd.Series(np.nan, index=series.index)
    return (values - values.mean()) / sd


def rolling_nine_structure(choice: np.ndarray, rewarded: np.ndarray) -> float:
    """Sample SD of valid nine-trial rolling win-switch rates."""
    n_trials = len(choice)
    win_switch = np.zeros(n_trials, dtype=float)
    win_stay = np.zeros(n_trials, dtype=float)
    for trial in range(1, n_trials):
        if rewarded[trial - 1] != 1:
            continue
        if choice[trial] != choice[trial - 1]:
            win_switch[trial] = 1
        else:
            win_stay[trial] = 1

    rolling_rates = []
    for start_index in range(n_trials - ROLLING_WINDOW + 1):
        stop_index = start_index + ROLLING_WINDOW
        denominator = (
            win_switch[start_index:stop_index].sum()
            + win_stay[start_index:stop_index].sum()
        )
        if denominator >= MIN_REWARDED_PER_WINDOW:
            rolling_rates.append(
                win_switch[start_index:stop_index].sum() / denominator
            )
    return (
        float(np.std(rolling_rates, ddof=1))
        if len(rolling_rates) >= MIN_VALID_WINDOWS
        else np.nan
    )


def session_features(session: pd.DataFrame) -> pd.Series:
    """Derive every publication behavioral marker from one 160-trial session."""
    ordered = session.sort_values("trial")
    choice = ordered["choice"].astype(str).to_numpy()
    rewarded = ordered["rewarded"].to_numpy(dtype=int)
    switch = (choice[1:] != choice[:-1]).astype(float)

    rewarded_transitions = rewarded[:-1] == 1
    wsr = switch[rewarded_transitions].mean()
    switch_rate = switch.mean()

    previous_switch, current_switch = switch[:-1], switch[1:]
    eligible_momentum = previous_switch == 1
    switch_momentum = (
        current_switch[eligible_momentum].mean()
        if eligible_momentum.sum() >= 5
        else np.nan
    )

    reward_run = np.zeros(len(choice), dtype=int)
    for trial in range(len(choice)):
        if rewarded[trial] == 1:
            continuing = (
                trial > 0
                and choice[trial] == choice[trial - 1]
                and rewarded[trial - 1] == 1
            )
            reward_run[trial] = reward_run[trial - 1] + 1 if continuing else 1

    run_switches = {run: 0 for run in range(1, 5)}
    run_opportunities = {run: 0 for run in range(1, 5)}
    for trial in range(len(choice) - 1):
        if rewarded[trial] != 1:
            continue
        bucket = min(reward_run[trial], 4)
        run_switches[bucket] += int(choice[trial + 1] != choice[trial])
        run_opportunities[bucket] += 1

    result: dict[str, float | int] = {
        "wsr": float(wsr),
        "switch_rate": float(switch_rate),
        "switch_momentum": float(switch_momentum),
        "structure": rolling_nine_structure(choice, rewarded),
    }
    run_probabilities = {}
    for run in range(1, 5):
        switches = run_switches[run]
        opportunities = run_opportunities[run]
        probability = switches / opportunities if opportunities >= 2 else np.nan
        result[f"k_run{run}"] = switches
        result[f"n_run{run}"] = opportunities
        result[f"p_run{run}"] = probability
        run_probabilities[run] = probability
    result["reward_sensitivity"] = (
        run_probabilities[1] - run_probabilities[4]
        if np.isfinite(run_probabilities[1]) and np.isfinite(run_probabilities[4])
        else np.nan
    )
    return pd.Series(result)


# Score R-GPTS persecution from the ten released items.
item_columns = [f"persecution_item_{index:02d}" for index in range(1, 11)]
items = weekly[item_columns].apply(pd.to_numeric, errors="coerce")
valid_items = items.notna().all(axis=1) & items.ge(0).all(axis=1) & items.le(4).all(axis=1)
weekly["rgpts"] = items.sum(axis=1).where(valid_items)
weekly["rgpts_complete"] = valid_items

# Derive the four rigidity facets and win-switch summaries from trial-level data.
behavior = (
    trials.groupby(["participant_id", "week"], sort=True)
    .apply(session_features, include_groups=False)
    .reset_index()
)
panel = weekly.merge(
    behavior, on=["participant_id", "week"], how="inner", validate="one_to_one"
)
panel = panel.dropna(subset=["rgpts", "wsr"]).copy()

for facet in FACETS:
    panel[f"z_{facet}"] = zscore_sample(panel[facet])
z_facets = [f"z_{facet}" for facet in FACETS]
panel["n_finite_facets"] = panel[z_facets].notna().sum(axis=1)
panel["rigidity"] = -panel[z_facets].mean(axis=1, skipna=True)
panel.loc[panel["n_finite_facets"] < 3, "rigidity"] = np.nan
panel = panel.sort_values(["participant_id", "week"]).reset_index(drop=True)

# Dataset locks prevent silent cohort, scoring or feature-definition drift.
assert len(panel) == N_PERSON_WEEKS
assert panel["participant_id"].nunique() == N_PEOPLE
assert panel[["participant_id", "week"]].duplicated().sum() == 0
assert panel["rgpts"].between(0, 40).all()
assert panel["structure"].notna().sum() == N_PERSON_WEEKS
assert panel["rigidity"].notna().sum() == N_PERSON_WEEKS
assert (panel["n_finite_facets"] >= 3).all()

panel_summary = pd.DataFrame(
    {
        "Quantity": [
            "Participants",
            "Person-weeks",
            "Eight-week completers",
            "Complete R-GPTS scores",
            "Rigidity scores",
        ],
        "Value": [
            panel["participant_id"].nunique(),
            len(panel),
            int((panel.groupby("participant_id").size() == 8).sum()),
            int(panel["rgpts_complete"].sum()),
            int(panel["rigidity"].notna().sum()),
        ],
    }
)
display(panel_summary)
display(panel[FACETS].notna().sum().rename("Available person-weeks").to_frame())

,Quantity,Value
0,Participants,150
1,Person-weeks,1000
2,Eight-week completers,107
3,Complete R-GPTS scores,1000
4,Rigidity scores,1000


,Available person-weeks
switch_rate,1000
switch_momentum,992
structure,1000
reward_sensitivity,991


## Main Figure 2: erratic, volatility-linked win-switching identifies paranoia

Panel a relates the trust curve to the supplied HGF volatility prior, \(\mu^0_3\). For each volatility tertile and reward-run length, switch counts and rewarded opportunities are pooled to plot the binomial proportion and Wilson 95% confidence interval. The displayed significance tests compare session-level probabilities in the highest and lowest volatility tertiles using the prespecified one-sided Mann–Whitney test.

Panel b averages win-switch rate within person, then compares participants below versus at or above the R-GPTS moderate persecution threshold of 11. The continuous person-level Spearman association is also calculated. The thresholded bars aid interpretation; the continuous analysis does not depend on this cut point.

In [4]:
def wilson_interval(k: float, n: float, z: float = 1.96) -> tuple[float, float, float]:
    """Wilson estimate and 95% interval for a binomial proportion."""
    if n <= 0:
        return np.nan, np.nan, np.nan
    proportion = k / n
    denominator = 1 + z**2 / n
    center = (proportion + z**2 / (2 * n)) / denominator
    half_width = z * np.sqrt(
        proportion * (1 - proportion) / n + z**2 / (4 * n**2)
    ) / denominator
    return proportion, center - half_width, center + half_width


figure2_data = panel.dropna(subset=["mu03"]).copy()
figure2_data["volatility_tertile"] = pd.qcut(
    figure2_data["mu03"], q=3, labels=["Low", "Middle", "High"]
)
run_labels = {1: "1", 2: "2", 3: "3", 4: "≥4"}
curve_rows, test_rows = [], []
for run in range(1, 5):
    for tertile in ["Low", "Middle", "High"]:
        group = figure2_data[figure2_data["volatility_tertile"] == tertile]
        switches = group[f"k_run{run}"].sum()
        opportunities = group[f"n_run{run}"].sum()
        estimate, lower, upper = wilson_interval(switches, opportunities)
        curve_rows.append(
            {
                "run": run,
                "run_label": run_labels[run],
                "tertile": tertile,
                "probability": estimate,
                "lower": lower,
                "upper": upper,
                "switches": int(switches),
                "opportunities": int(opportunities),
            }
        )

    low = figure2_data.loc[
        figure2_data["volatility_tertile"] == "Low", f"p_run{run}"
    ].dropna()
    high = figure2_data.loc[
        figure2_data["volatility_tertile"] == "High", f"p_run{run}"
    ].dropna()
    comparison = stats.mannwhitneyu(high, low, alternative="greater")
    test_rows.append(
        {
            "Reward run": run_labels[run],
            "U": comparison.statistic,
            "P": comparison.pvalue,
            "n high": len(high),
            "n low": len(low),
        }
    )

figure2_curve = pd.DataFrame(curve_rows)
figure2a_tests = pd.DataFrame(test_rows)
person = panel.groupby("participant_id", as_index=False).agg(
    rgpts=("rgpts", "mean"), wsr=("wsr", "mean")
)
person["group"] = np.where(
    person["rgpts"] >= PARANOIA_THRESHOLD, "At/above 11", "Below 11"
)
low_person = person.loc[person["group"] == "Below 11", "wsr"]
high_person = person.loc[person["group"] == "At/above 11", "wsr"]
figure2b_test = stats.mannwhitneyu(high_person, low_person, alternative="greater")
figure2b_rho = stats.spearmanr(person["wsr"], person["rgpts"])

fig, axes = plt.subplots(
    1, 2, figsize=(11.2, 4.9), gridspec_kw={"width_ratios": [1.25, 1.0]}
)
palette = {"Low": "#C5D0D6", "Middle": "#7A929E", "High": "#3D5563"}
for tertile in ["High", "Middle", "Low"]:
    data = figure2_curve[figure2_curve["tertile"] == tertile]
    x = data["run"].to_numpy()
    y = 100 * data["probability"].to_numpy()
    lower_error = 100 * (data["probability"] - data["lower"]).to_numpy()
    upper_error = 100 * (data["upper"] - data["probability"]).to_numpy()
    axes[0].errorbar(
        x,
        y,
        yerr=[lower_error, upper_error],
        marker="o",
        lw=2.6,
        capsize=2.5,
        label=tertile,
        color=palette[tertile],
    )
axes[0].set(
    xlabel="Consecutive rewards from the chosen deck",
    ylabel="P(abandon the deck | rewarded) (%)",
    xticks=[1, 2, 3, 4],
    xticklabels=["1", "2", "3", "≥4"],
    ylim=(0, 22),
)
axes[0].legend(title=r"Volatility prior $\mu^0_3$", frameon=False)
axes[0].set_title("a  Volatility belief aligns with win-switching", loc="left", fontweight="bold")

order = ["Below 11", "At/above 11"]
means = person.groupby("group")["wsr"].mean().reindex(order)
sems = person.groupby("group")["wsr"].sem().reindex(order)
axes[1].bar(
    [0, 1],
    100 * means,
    yerr=100 * sems,
    color=["#5B8FA8", "#C47B7B"],
    width=0.62,
    capsize=5,
)
axes[1].set_xticks(
    [0, 1],
    [f"Below 11\n(n = {len(low_person)})", f"At/above 11\n(n = {len(high_person)})"],
)
axes[1].set(ylabel="P(abandon the deck | rewarded) (%)", ylim=(0, 22))
axes[1].set_title("b  Win-switch rate identifies paranoia", loc="left", fontweight="bold")

fig.suptitle(
    "Figure 2 | Erratic, volatility-linked win-switching identifies paranoia",
    fontweight="bold",
)
fig.tight_layout()
fig.savefig(OUT / "figure2_main.png", bbox_inches="tight")
fig.savefig(OUT / "figure2_main.pdf", bbox_inches="tight")
plt.show()

figure2_group_summary = pd.DataFrame(
    {
        "Group": order,
        "n": [len(low_person), len(high_person)],
        "Mean WSR (%)": 100 * means.to_numpy(),
        "SEM (%)": 100 * sems.to_numpy(),
    }
)
display(figure2a_tests.style.format({"U": "{:.1f}", "P": "{:.4g}"}))
display(figure2_group_summary.style.format({"Mean WSR (%)": "{:.2f}", "SEM (%)": "{:.2f}"}))
print(f"Group comparison: U={figure2b_test.statistic:.1f}, one-sided P={figure2b_test.pvalue:.4g}")
print(f"Continuous association: Spearman ρ={figure2b_rho.statistic:.3f}, P={figure2b_rho.pvalue:.4g}")

assert len(low_person) == 127 and len(high_person) == 23
assert high_person.mean() > 0.14 and low_person.mean() < 0.06
assert figure2b_test.pvalue < 0.001

C:\Users\ps967\AppData\Local\Temp\1\ipykernel_31596\1186904634.py:124: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Reward run,U,P,n high,n low
0,1,75745.0,2.075e-17,334,334
1,2,72577.5,2.788e-16,333,334
2,3,65517.0,7.538e-10,327,334
3,≥4,71711.0,8.805e-15,325,334


,Group,n,Mean WSR (%),SEM (%)
0,Below 11,127,5.03,0.87
1,At/above 11,23,14.45,4.40


Group comparison: U=2098.0, one-sided P=0.0004459
Continuous association: Spearman ρ=0.287, P=0.0003787


## Main Figure 3: win-switch rigidity tracks within-person changes

This descriptive visualization includes participants who completed all eight sessions and had sufficient persecution variation to order their weeks (within-person s.d. ≥ 0.5). Weeks are sorted separately for each participant from lowest to highest persecution, with calendar week breaking ties, then expressed in within-person standard-deviation units.

The accompanying two-sided paired tests compare each participant's mean across ranks 1–2 with their mean across ranks 7–8. The full-panel mixed models in Figure 4 are the primary inferential within-person analyses; this reordering plot illustrates their behavioral meaning.

In [5]:
def person_standardize(values: pd.Series, ids: pd.Series) -> pd.Series:
    means = values.groupby(ids).transform("mean")
    standard_deviations = values.groupby(ids).transform("std").replace(0, np.nan)
    return (values - means) / standard_deviations


def build_ranked_weeks(data: pd.DataFrame) -> pd.DataFrame:
    counts = data.groupby("participant_id").size()
    symptom_sd = data.groupby("participant_id")["rgpts"].std()
    eligible = counts.index[(counts == 8) & (symptom_sd >= 0.5)]

    ranked_groups = []
    for _, group in data[data["participant_id"].isin(eligible)].groupby("participant_id"):
        ordered = group.sort_values(["rgpts", "week"]).copy()
        ordered["rank"] = np.arange(1, 9)
        ranked_groups.append(ordered)
    ranked_data = pd.concat(ranked_groups, ignore_index=True)

    for column in ["rgpts", "wsr", "rigidity"]:
        ranked_data[f"z_{column}_person"] = person_standardize(
            ranked_data[column], ranked_data["participant_id"]
        )
    return ranked_data


def extreme_rank_test(data: pd.DataFrame, column: str) -> dict[str, float]:
    low = data[data["rank"] <= 2].groupby("participant_id")[column].mean()
    high = data[data["rank"] >= 7].groupby("participant_id")[column].mean()
    paired = pd.concat({"low": low, "high": high}, axis=1).dropna()
    test = stats.ttest_rel(paired["high"], paired["low"])
    difference = paired["high"] - paired["low"]
    confidence_interval = stats.t.interval(
        0.95,
        len(difference) - 1,
        loc=difference.mean(),
        scale=stats.sem(difference),
    )
    return {
        "n": len(paired),
        "mean_difference": difference.mean(),
        "ci_low": confidence_interval[0],
        "ci_high": confidence_interval[1],
        "t": test.statistic,
        "df": len(paired) - 1,
        "p": test.pvalue,
    }


ranked = build_ranked_weeks(panel)
assert ranked["participant_id"].nunique() == 68
ranked_summary = (
    ranked.groupby("rank")
    .agg(
        persecution=("z_rgpts_person", "mean"),
        persecution_sem=("z_rgpts_person", "sem"),
        wsr=("z_wsr_person", "mean"),
        wsr_sem=("z_wsr_person", "sem"),
        rigidity=("z_rigidity_person", "mean"),
        rigidity_sem=("z_rigidity_person", "sem"),
    )
    .reset_index()
)
figure3_wsr = extreme_rank_test(ranked, "z_wsr_person")
figure3_rigidity = extreme_rank_test(ranked, "z_rigidity_person")

fig, axes = plt.subplots(3, 1, figsize=(7.2, 6.4), sharex=True)
series = [
    ("persecution", "persecution_sem", "Paranoia", COLORS["symptom"]),
    ("wsr", "wsr_sem", "Win-switch rate", COLORS["wsr"]),
    ("rigidity", "rigidity_sem", "Win-switch rigidity", COLORS["rigidity"]),
]
for label, (value, error, ylabel, color), axis in zip("abc", series, axes):
    x = ranked_summary["rank"].to_numpy()
    y = ranked_summary[value].to_numpy()
    standard_error = ranked_summary[error].to_numpy()
    axis.plot(x, y, marker="o", color=color, lw=2.5)
    axis.fill_between(
        x, y - standard_error, y + standard_error, color=color, alpha=0.16, linewidth=0
    )
    axis.axhline(0, color="#D5D2CC", ls="--", lw=1)
    axis.set_ylabel(f"{ylabel}\n(within-person s.d.)", color=color)
    axis.set_title(label, loc="left", fontweight="bold")
axes[1].set_ylim(-0.45, 0.45)
axes[2].set_ylim(-0.45, 0.45)
axes[-1].set(
    xlabel="Week reordered within participant: lowest → highest persecution",
    xticks=np.arange(1, 9),
)
fig.suptitle(
    "Figure 3 | Win-switch rigidity tracks within-person changes",
    fontweight="bold",
)
fig.tight_layout()
fig.savefig(OUT / "figure3_main.png", bbox_inches="tight")
fig.savefig(OUT / "figure3_main.pdf", bbox_inches="tight")
plt.show()

figure3_tests = pd.DataFrame(
    [
        {"Marker": "Win-switch rate", **figure3_wsr},
        {"Marker": "Win-switch rigidity", **figure3_rigidity},
    ]
)
display(
    figure3_tests.style.format(
        {
            "mean_difference": "{:.3f}",
            "ci_low": "{:.3f}",
            "ci_high": "{:.3f}",
            "t": "{:.3f}",
            "df": "{:.0f}",
            "p": "{:.4f}",
        }
    )
)

assert figure3_wsr["n"] == 67 and np.isclose(figure3_wsr["p"], 0.493, atol=0.002)
assert figure3_rigidity["n"] == 68 and np.isclose(figure3_rigidity["p"], 0.0115, atol=0.001)

C:\Users\ps967\AppData\Local\Temp\1\ipykernel_31596\3019777539.py:96: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Marker,n,mean_difference,ci_low,ci_high,t,df,p
0,Win-switch rate,67,-0.086,-0.335,0.163,-0.690,66,0.4928
1,Win-switch rigidity,68,0.321,0.074,0.568,2.597,67,0.0115


## Main Figure 4: identification and tracking require distinct signatures

Each marker is standardized across the 1,000 person-weeks. A participant's mean standardized value is the between-person component; each week's departure from that participant mean is the within-person component. Separate participant-random-intercept models estimate both components simultaneously while controlling categorical week effects.

The figure is drawn directly from the fitted models. Two-sided Wald tests contrast the between- and within-person coefficients. A separate joint model is used only to compare the two within-person marker coefficients; it is not the source of the main bars.

In [6]:
def decompose_marker(
    data: pd.DataFrame, marker: str, prefix: str = "marker"
) -> pd.DataFrame:
    """Add standardized between- and within-person marker components."""
    decomposed = data.dropna(subset=[marker, "rgpts"]).copy()
    decomposed[f"{prefix}_z"] = zscore_sample(decomposed[marker])
    decomposed[f"{prefix}_between"] = decomposed.groupby("participant_id")[
        f"{prefix}_z"
    ].transform("mean")
    decomposed[f"{prefix}_within"] = (
        decomposed[f"{prefix}_z"] - decomposed[f"{prefix}_between"]
    )
    return decomposed


def fit_between_within(data: pd.DataFrame, marker: str):
    model_data = decompose_marker(data, marker)
    model = smf.mixedlm(
        "rgpts ~ marker_within + marker_between + C(week)",
        model_data,
        groups=model_data["participant_id"],
    ).fit(reml=False)
    return model, model_data


def coefficient_row(model, term: str, marker: str, level: str) -> dict[str, float | str]:
    estimate = float(model.params[term])
    standard_error = float(model.bse[term])
    return {
        "Marker": marker,
        "Level": level,
        "b": estimate,
        "SE": standard_error,
        "CI low": estimate - 1.96 * standard_error,
        "CI high": estimate + 1.96 * standard_error,
        "z": estimate / standard_error,
        "P": float(model.pvalues[term]),
        "Person-weeks": int(model.nobs),
        "Participants": int(model.model.groups.shape[0] and len(np.unique(model.model.groups))),
    }


def wald_difference(model, first: str, second: str) -> dict[str, float]:
    covariance = model.cov_params()
    difference = float(model.params[first] - model.params[second])
    variance = float(
        covariance.loc[first, first]
        + covariance.loc[second, second]
        - 2 * covariance.loc[first, second]
    )
    standard_error = np.sqrt(variance)
    z_value = difference / standard_error
    return {
        "difference": difference,
        "SE": standard_error,
        "CI low": difference - 1.96 * standard_error,
        "CI high": difference + 1.96 * standard_error,
        "z": z_value,
        "P": 2 * stats.norm.sf(abs(z_value)),
    }


wsr_model, wsr_model_data = fit_between_within(panel, "wsr")
rigidity_model, rigidity_model_data = fit_between_within(panel, "rigidity")
figure4_coefficients = pd.DataFrame(
    [
        coefficient_row(wsr_model, "marker_between", "Win-switch rate", "Between people"),
        coefficient_row(wsr_model, "marker_within", "Win-switch rate", "Within person"),
        coefficient_row(
            rigidity_model, "marker_between", "Win-switch rigidity", "Between people"
        ),
        coefficient_row(
            rigidity_model, "marker_within", "Win-switch rigidity", "Within person"
        ),
    ]
)
figure4_contrasts = pd.DataFrame(
    [
        {
            "Marker": "Win-switch rate",
            **wald_difference(wsr_model, "marker_between", "marker_within"),
        },
        {
            "Marker": "Win-switch rigidity",
            **wald_difference(rigidity_model, "marker_between", "marker_within"),
        },
    ]
)

# Joint model used only for the direct comparison of within-person coefficients.
joint = panel.dropna(subset=["rgpts", "wsr", "rigidity"]).copy()
for marker in ["wsr", "rigidity"]:
    joint[f"{marker}_z"] = zscore_sample(joint[marker])
    joint[f"{marker}_between"] = joint.groupby("participant_id")[
        f"{marker}_z"
    ].transform("mean")
    joint[f"{marker}_within"] = joint[f"{marker}_z"] - joint[f"{marker}_between"]
joint_model = smf.mixedlm(
    "rgpts ~ wsr_within + wsr_between + rigidity_within + rigidity_between + C(week)",
    joint,
    groups=joint["participant_id"],
).fit(reml=False)
joint_within_contrast = wald_difference(
    joint_model, "rigidity_within", "wsr_within"
)
marker_correlation = stats.spearmanr(panel["wsr"], panel["rigidity"])

levels = ["Between people", "Within person"]
markers = ["Win-switch rate", "Win-switch rigidity"]
fig, axis = plt.subplots(figsize=(7.3, 4.8))
x = np.arange(len(levels))
width = 0.30
for offset, marker, color in [
    (-width / 2, markers[0], COLORS["wsr"]),
    (width / 2, markers[1], COLORS["rigidity"]),
]:
    data = (
        figure4_coefficients.set_index(["Marker", "Level"])
        .loc[marker]
        .reindex(levels)
    )
    axis.bar(
        x + offset,
        data["b"],
        width,
        yerr=1.96 * data["SE"],
        capsize=5,
        label=marker,
        color=color,
    )
axis.axhline(0, color="#777777", lw=1)
axis.set_xticks(
    x,
    [
        "Between people\n(who is more paranoid?)",
        "Within person\n(which weeks are more paranoid?)",
    ],
)
axis.set_ylabel("Association with R-GPTS persecution\n(b per marker s.d., 95% CI; week-adjusted)")
axis.legend(frameon=False)
axis.set_title(
    "Figure 4 | Identification and tracking require distinct signatures",
    fontweight="bold",
)
fig.tight_layout()
fig.savefig(OUT / "figure4_main.png", bbox_inches="tight")
fig.savefig(OUT / "figure4_main.pdf", bbox_inches="tight")
plt.show()

display(
    figure4_coefficients.style.format(
        {"b": "{:+.3f}", "SE": "{:.3f}", "CI low": "{:+.3f}", "CI high": "{:+.3f}", "P": "{:.4f}"}
    )
)
display(
    figure4_contrasts.style.format(
        {"difference": "{:+.3f}", "SE": "{:.3f}", "CI low": "{:+.3f}", "CI high": "{:+.3f}", "z": "{:+.3f}", "P": "{:.4f}"}
    )
)
print(f"Session-level WSR–rigidity Spearman ρ={marker_correlation.statistic:.3f}, P={marker_correlation.pvalue:.4g}")
print(
    "Joint-model contrast, rigidity within minus WSR within: "
    f"Δb={joint_within_contrast['difference']:+.3f}, "
    f"z={joint_within_contrast['z']:.3f}, P={joint_within_contrast['P']:.3f}"
)

locked = figure4_coefficients.set_index(["Marker", "Level"])
assert np.isclose(locked.loc[("Win-switch rate", "Between people"), "b"], 1.742, atol=0.01)
assert np.isclose(locked.loc[("Win-switch rate", "Within person"), "b"], 0.090, atol=0.01)
assert np.isclose(locked.loc[("Win-switch rigidity", "Between people"), "b"], -2.226, atol=0.02)
assert np.isclose(locked.loc[("Win-switch rigidity", "Within person"), "b"], 0.494, atol=0.01)

C:\Users\ps967\AppData\Local\Temp\1\ipykernel_31596\3537729498.py:148: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Marker,Level,b,SE,CI low,CI high,z,P,Person-weeks,Participants
0,Win-switch rate,Between people,+1.742,0.548,+0.668,+2.816,3.180522,0.0015,1000,150
1,Win-switch rate,Within person,+0.090,0.220,-0.341,+0.522,0.410304,0.6816,1000,150
2,Win-switch rigidity,Between people,-2.226,0.673,-3.544,-0.907,-3.309266,0.0009,1000,150
3,Win-switch rigidity,Within person,+0.494,0.196,+0.110,+0.877,2.521227,0.0117,1000,150


,Marker,difference,SE,CI low,CI high,z,P
0,Win-switch rate,+1.652,0.588,+0.498,+2.805,+2.807,0.0050
1,Win-switch rigidity,-2.719,0.699,-4.089,-1.349,-3.891,0.0001


Session-level WSR–rigidity Spearman ρ=-0.725, P=1.739e-163
Joint-model contrast, rigidity within minus WSR within: Δb=+0.142, z=0.628, P=0.530


## Supplementary Table 1: sample characteristics

The public input contains only the aggregate values displayed in the final demographic table. Individual demographic records are deliberately excluded from the reproducibility package because they are unnecessary for the analyses and would increase disclosure risk. Counts and percentages use the 150-person primary cohort as the denominator.

In [7]:
supplementary_table_1 = demographics_source.copy()
required_characteristics = {
    "Age, years",
    "Gender",
    "Race",
    "Hispanic or Latino ethnicity",
    "Education",
    "Employment",
    "Annual income, US$",
}
assert required_characteristics.issubset(
    set(supplementary_table_1["Characteristic"].dropna())
)

display(supplementary_table_1)
supplementary_table_1.to_csv(
    OUT / "supplementary_table_1_sample_characteristics.csv", index=False
)

,Characteristic,Category or statistic,Value
0,"Age, years",Mean (s.d.),39.7 (11.2)
1,NaN,Median (IQR),37.0 (31.0–47.8)
2,NaN,Range,21–73
3,Gender,Female,78 (52.0%)
4,NaN,Male,71 (47.3%)
5,NaN,Multiple selections (female and male),1 (0.7%)
6,Race,American Indian or Alaska Native,1 (0.7%)
7,NaN,Asian,4 (2.7%)
8,NaN,Black or African American,16 (10.7%)
9,NaN,Native Hawaiian or Other Pacific Islander,1 (0.7%)


## Supplementary Table 2: retention and analysis-specific samples

Panel a reports weekly questionnaire completion and paired PRL/R-GPTS inclusion. One completed week-2 questionnaire lacked a matching usable PRL session, yielding 1,000 paired person-weeks. Panel b defines each analysis-specific subset used in the manuscript.

The notebook independently reconstructs all sample sizes available from the public data and checks them against the publication-formatted table source.

In [8]:
supplementary_table_2a = retention_source.copy()
supplementary_table_2b = analysis_samples_source.copy()

# Reconstruct public-data quantities and compare them with the formatted tables.
paired_by_week = panel.groupby("week").size().reindex(range(1, 9)).to_numpy()
assert np.array_equal(
    paired_by_week,
    supplementary_table_2a["Paired panel"].to_numpy(dtype=int),
)
assert supplementary_table_2a["Finished"].sum() == 1_001
assert supplementary_table_2a["R-GPTS"].sum() == 1_001
assert supplementary_table_2a["Paired panel"].sum() == N_PERSON_WEEKS

computed_samples = {
    "Primary panel": (panel["participant_id"].nunique(), len(panel)),
    "Eight-week completers": (
        int((panel.groupby("participant_id").size() == 8).sum()),
        int((panel.groupby("participant_id").size() == 8).sum() * 8),
    ),
    "Ranked-week illustration": (
        ranked["participant_id"].nunique(),
        len(ranked),
    ),
    "Fluctuator subgroup": (
        panel.loc[
            panel.groupby("participant_id")["rgpts"].transform("std") > 2,
            "participant_id",
        ].nunique(),
        int((panel.groupby("participant_id")["rgpts"].transform("std") > 2).sum()),
    ),
    "Omicron window": (
        panel.dropna(subset=["omicron_concern"])["participant_id"].nunique(),
        len(panel.dropna(subset=["omicron_concern"])),
    ),
}
for row in supplementary_table_2b.itertuples(index=False):
    people, person_weeks = computed_samples[row.Sample]
    assert people == int(row.People)
    assert person_weeks == int(str(row._3).replace(",", ""))

display(supplementary_table_2a)
display(supplementary_table_2b)
supplementary_table_2a.to_csv(
    OUT / "supplementary_table_2_weekly_retention.csv", index=False
)
supplementary_table_2b.to_csv(
    OUT / "supplementary_table_2_analysis_samples.csv", index=False
)

,Week,Finished,R-GPTS,Paired panel,Retained
0,1,150,150,150,100.0%
1,2,140,140,139,92.7%
2,3,130,130,130,86.7%
3,4,126,126,126,84.0%
4,5,120,120,120,80.0%
5,6,116,116,116,77.3%
6,7,111,111,111,74.0%
7,8,108,108,108,72.0%


,Sample,Definition,People,Person-weeks
0,Primary panel,Primary mixed models,150,"1,000"
1,Eight-week completers,Completed all sessions,107,856
2,Ranked-week illustration,Completers; R-GPTS s.d. ≥0.5,68,544
3,Fluctuator subgroup,Within-person R-GPTS s.d. >2,52,368
4,Omicron window,Weeks 3–8 with concern item,131,708


## Supplementary Figure 1: reliability and robustness of win-switch rigidity

Panel a reconstructs two rigidity half-scores from every session. Switchiness, switch momentum and reward sensitivity use alternating eligible events; temporal structure is recomputed over trials 1–80 and 81–160. The Spearman correlation between halves is corrected to full-session length as \(r_{SB}=2\rho/(1+\rho)\), both overall and separately by week.

Panel b rebuilds rigidity after omitting each facet in turn. Panel c demeans rigidity and persecution by person and week, then shuffles rigidity across weeks within participant 5,000 times. These checks test whether the association depends on one facet or on the observed temporal alignment. They do not establish a large, causal or person-specific effect.

In [9]:
def spearman_brown(rho: float) -> float:
    return 2 * rho / (1 + rho) if np.isfinite(rho) and rho > -1 else np.nan


def reward_sensitivity_halves(
    choice: np.ndarray, rewarded: np.ndarray
) -> tuple[float, float]:
    reward_run = np.zeros(len(choice), dtype=int)
    for trial in range(len(choice)):
        if rewarded[trial] == 1:
            continuing = (
                trial > 0
                and choice[trial] == choice[trial - 1]
                and rewarded[trial - 1] == 1
            )
            reward_run[trial] = reward_run[trial - 1] + 1 if continuing else 1

    runs, switches = [], []
    for trial in range(len(choice) - 1):
        if rewarded[trial] == 1:
            runs.append(reward_run[trial])
            switches.append(int(choice[trial + 1] != choice[trial]))
    runs = np.asarray(runs)
    switches = np.asarray(switches, dtype=float)

    def score(mask: np.ndarray) -> float:
        selected_runs, selected_switches = runs[mask], switches[mask]
        after_one = selected_switches[selected_runs == 1]
        after_four = selected_switches[selected_runs >= 4]
        if len(after_one) < 2 or len(after_four) < 2:
            return np.nan
        return float(after_one.mean() - after_four.mean())

    alternating = np.arange(len(runs)) % 2 == 0
    return score(alternating), score(~alternating)


def split_half_facets(session: pd.DataFrame) -> pd.Series:
    ordered = session.sort_values("trial").reset_index(drop=True)
    choice = ordered["choice"].astype(str).to_numpy()
    rewarded = ordered["rewarded"].to_numpy(dtype=int)
    switch = (choice[1:] != choice[:-1]).astype(float)

    switch_rate_a = switch[0::2].mean()
    switch_rate_b = switch[1::2].mean()

    momentum_outcomes = switch[1:][switch[:-1] == 1]
    if len(momentum_outcomes) >= 5:
        momentum_a = momentum_outcomes[0::2].mean()
        momentum_b = momentum_outcomes[1::2].mean()
    else:
        momentum_a = momentum_b = np.nan

    sensitivity_a, sensitivity_b = reward_sensitivity_halves(choice, rewarded)
    midpoint = len(ordered) // 2
    structure_a = rolling_nine_structure(choice[:midpoint], rewarded[:midpoint])
    structure_b = rolling_nine_structure(choice[midpoint:], rewarded[midpoint:])

    return pd.Series(
        {
            "switch_rate_a": switch_rate_a,
            "switch_rate_b": switch_rate_b,
            "switch_momentum_a": momentum_a,
            "switch_momentum_b": momentum_b,
            "structure_a": structure_a,
            "structure_b": structure_b,
            "reward_sensitivity_a": sensitivity_a,
            "reward_sensitivity_b": sensitivity_b,
        }
    )


halves = (
    trials.groupby(["participant_id", "week"], sort=True)
    .apply(split_half_facets, include_groups=False)
    .reset_index()
)
assert len(halves) == N_PERSON_WEEKS
for half in ["a", "b"]:
    half_columns = [f"{facet}_{half}" for facet in FACETS]
    for column in half_columns:
        halves[f"z_{column}"] = zscore_sample(halves[column])
    halves[f"rigidity_{half}"] = -halves[
        [f"z_{column}" for column in half_columns]
    ].mean(axis=1)

required_half_facets = [f"{facet}_{half}" for facet in FACETS for half in ["a", "b"]]
complete_halves = halves.dropna(subset=required_half_facets).copy()
rigidity_half_rho = stats.spearmanr(
    complete_halves["rigidity_a"], complete_halves["rigidity_b"]
).statistic
rigidity_sb = spearman_brown(rigidity_half_rho)
weekly_reliability_rows = []
for week, data in complete_halves.groupby("week"):
    rho = stats.spearmanr(data["rigidity_a"], data["rigidity_b"]).statistic
    weekly_reliability_rows.append(
        {"Week": week, "n": len(data), "r_SB": spearman_brown(rho)}
    )
weekly_reliability = pd.DataFrame(weekly_reliability_rows)


def unbalanced_one_way_icc(data: pd.DataFrame, value: str) -> float:
    rows = [
        group[value].dropna().to_numpy(float)
        for _, group in data.groupby("participant_id")
    ]
    rows = [row for row in rows if len(row) >= 2]
    mean_sessions = np.mean([len(row) for row in rows])
    grand_mean = np.mean(np.concatenate(rows))
    between_ms = sum(
        len(row) * (row.mean() - grand_mean) ** 2 for row in rows
    ) / (len(rows) - 1)
    within_ms = sum(
        ((row - row.mean()) ** 2).sum() for row in rows
    ) / sum(len(row) - 1 for row in rows)
    return float(
        (between_ms - within_ms)
        / (between_ms + (mean_sessions - 1) * within_ms)
    )


rigidity_icc = unbalanced_one_way_icc(
    panel[["participant_id", "week", "rigidity"]], "rigidity"
)
wsr_icc_model = smf.mixedlm(
    "wsr ~ 1", panel, groups=panel["participant_id"]
).fit(reml=True)
wsr_between_variance = float(wsr_icc_model.cov_re.iloc[0, 0])
wsr_icc = wsr_between_variance / (
    wsr_between_variance + float(wsr_icc_model.scale)
)


def permutation_correlation(
    data: pd.DataFrame,
    rigidity_column: str = "rigidity",
    n_permutations: int = 5_000,
    seed: int = 0,
    return_null: bool = False,
):
    permutation_data = data.dropna(subset=[rigidity_column, "rgpts"]).copy()
    symptom = zscore_sample(permutation_data["rgpts"])
    rigidity = zscore_sample(permutation_data[rigidity_column])
    symptom -= symptom.groupby(permutation_data["participant_id"]).transform("mean")
    symptom -= symptom.groupby(permutation_data["week"]).transform("mean")
    rigidity -= rigidity.groupby(permutation_data["participant_id"]).transform("mean")
    rigidity -= rigidity.groupby(permutation_data["week"]).transform("mean")

    valid = symptom.notna() & rigidity.notna()
    symptom_values = symptom[valid].to_numpy()
    rigidity_values = rigidity[valid].to_numpy()
    ids = permutation_data.loc[valid, "participant_id"].to_numpy()
    observed = float(stats.spearmanr(rigidity_values, symptom_values).statistic)

    rng = np.random.default_rng(seed)
    groups = [np.flatnonzero(ids == participant) for participant in np.unique(ids)]
    null = np.empty(n_permutations)
    for iteration in range(n_permutations):
        permuted = rigidity_values.copy()
        for indices in groups:
            permuted[indices] = rng.permutation(permuted[indices])
        null[iteration] = stats.spearmanr(permuted, symptom_values).statistic
    p_value = (np.sum(np.abs(null) >= abs(observed)) + 1) / (n_permutations + 1)

    result = {
        "rho": observed,
        "P": float(p_value),
        "Person-weeks": int(valid.sum()),
        "Participants": int(permutation_data.loc[valid, "participant_id"].nunique()),
        "Permutations": n_permutations,
    }
    return (result, null) if return_null else result


permutation_result, permutation_null = permutation_correlation(
    panel, n_permutations=5_000, seed=0, return_null=True
)
facet_labels = {
    "switch_rate": "Switchiness",
    "switch_momentum": "Stickiness",
    "structure": "Structure",
    "reward_sensitivity": "Reward sensitivity",
}
leave_one_out_rows = []
for dropped_facet in FACETS:
    retained = [f"z_{facet}" for facet in FACETS if facet != dropped_facet]
    temporary = panel.copy()
    temporary["rigidity_leave_one_out"] = -temporary[retained].mean(
        axis=1, skipna=True
    )
    leave_one_out_rows.append(
        {
            "Facet removed": facet_labels[dropped_facet],
            **permutation_correlation(
                temporary,
                rigidity_column="rigidity_leave_one_out",
                n_permutations=2_000,
                seed=1,
            ),
        }
    )
leave_one_out = pd.DataFrame(leave_one_out_rows)

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.25))
fig.subplots_adjust(left=0.065, right=0.985, top=0.80, bottom=0.18, wspace=0.48)

axes[0].bar(
    weekly_reliability["Week"],
    weekly_reliability["r_SB"],
    color=COLORS["blue"],
    width=0.68,
)
axes[0].axhline(rigidity_sb, color=COLORS["dark"], ls="--", lw=1.2)
axes[0].set(
    xlabel="Weekly session",
    ylabel="Spearman–Brown reliability",
    xticks=weekly_reliability["Week"],
    ylim=(0, 1),
)
axes[0].text(
    0.98,
    rigidity_sb + 0.025,
    f"Overall = {rigidity_sb:.2f}",
    transform=axes[0].get_yaxis_transform(),
    ha="right",
    fontsize=8.5,
)
axes[0].set_title("a  Four-facet split-half reliability", loc="left", fontweight="bold")

leave_one_out_display = leave_one_out.iloc[::-1].reset_index(drop=True)
y = np.arange(len(leave_one_out_display))
axes[1].axvline(0, color="#999999", lw=1)
axes[1].scatter(leave_one_out_display["rho"], y, color=COLORS["rigidity"], s=45)
for index, row in leave_one_out_display.iterrows():
    axes[1].text(row["rho"] + 0.004, index, f"P={row['P']:.3f}", va="center", fontsize=8.5)
axes[1].set_yticks(y, leave_one_out_display["Facet removed"], fontsize=9)
axes[1].set(xlim=(0, 0.13), xlabel="Within-person Spearman ρ")
axes[1].set_title("b  Leave-one-facet-out checks", loc="left", fontweight="bold")

axes[2].hist(permutation_null, bins=35, color=COLORS["blue"], alpha=0.80)
axes[2].axvline(permutation_result["rho"], color=COLORS["rigidity"], lw=2.3)
axes[2].axvline(-permutation_result["rho"], color=COLORS["rigidity"], lw=1.2, ls=":")
axes[2].text(
    0.04,
    0.95,
    f"Observed ρ={permutation_result['rho']:.3f}\nPermutation P={permutation_result['P']:.3f}",
    transform=axes[2].transAxes,
    va="top",
    fontsize=9,
)
axes[2].set(xlabel="Permuted within-person Spearman ρ", ylabel="Permutations")
axes[2].set_title("c  Within-person permutation test", loc="left", fontweight="bold")
fig.suptitle(
    "Supplementary Fig. 1 | Reliability and robustness of win-switch rigidity",
    fontweight="bold",
)
fig.savefig(OUT / "supplementary_figure_1.png", bbox_inches="tight")
fig.savefig(OUT / "supplementary_figure_1.pdf", bbox_inches="tight")
plt.show()

reliability_results = pd.DataFrame(
    [
        {"Marker": "Win-switch rate", "Statistic": "Between-week ICC", "Estimate": wsr_icc},
        {"Marker": "Win-switch rigidity", "Statistic": "Spearman–Brown split-half", "Estimate": rigidity_sb},
        {"Marker": "Win-switch rigidity", "Statistic": "Between-week ICC", "Estimate": rigidity_icc},
    ]
)
display(reliability_results.style.format({"Estimate": "{:.3f}"}))
display(weekly_reliability.style.format({"r_SB": "{:.3f}"}))
display(leave_one_out.style.format({"rho": "{:+.3f}", "P": "{:.4f}"}))
display(pd.DataFrame([permutation_result]).style.format({"rho": "{:+.3f}", "P": "{:.4f}"}))

weekly_reliability.to_csv(OUT / "figure_s1a_weekly_reliability.csv", index=False)
leave_one_out.to_csv(OUT / "figure_s1b_leave_one_facet_out.csv", index=False)
pd.DataFrame({"permuted_rho": permutation_null}).to_csv(
    OUT / "figure_s1c_permutation_null.csv", index=False
)

assert np.isclose(rigidity_sb, 0.829548, atol=0.00001)
assert np.isclose(rigidity_icc, 0.538, atol=0.01)
assert np.isclose(wsr_icc, 0.713, atol=0.01)
assert np.isclose(permutation_result["P"], 0.011, atol=0.002)
assert round(leave_one_out["P"].min(), 3) == 0.003
assert round(leave_one_out["P"].max(), 3) == 0.038

C:\Users\ps967\AppData\Local\Temp\1\ipykernel_31596\3375897389.py:259: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Marker,Statistic,Estimate
0,Win-switch rate,Between-week ICC,0.713
1,Win-switch rigidity,Spearman–Brown split-half,0.830
2,Win-switch rigidity,Between-week ICC,0.538


,Week,n,r_SB
0,1,143,0.843
1,2,138,0.859
2,3,128,0.867
3,4,124,0.703
4,5,117,0.853
5,6,112,0.793
6,7,109,0.865
7,8,105,0.759


,Facet removed,rho,P,Person-weeks,Participants,Permutations
0,Switchiness,+0.082,0.0185,1000,150,2000
1,Stickiness,+0.082,0.0185,1000,150,2000
2,Structure,+0.101,0.0030,1000,150,2000
3,Reward sensitivity,+0.074,0.0385,1000,150,2000


,rho,P,Person-weeks,Participants,Permutations
0,+0.090,0.0106,1000,150,5000


## Supplementary Figure 2: sensitivity to repeated testing and Omicron

Panel a reports the prevalence of Omicron concern during weeks 3–8. Panel b compares the primary within-person rigidity estimate with four sensitivity specifications: participant-specific linear session slopes, exclusion of week 1, restriction to weeks 3–8, and adjustment for between- and within-person Omicron concern in that same restricted sample.

Categorical week effects in the primary model already absorb any sample-wide nonlinear practice or calendar trajectory. Restricted-sample models re-standardize and re-center rigidity within the retained observations, so attenuation after removing weeks is not equivalent to covariate adjustment.

In [10]:
def fit_random_session_slope(data: pd.DataFrame, marker: str):
    model_data = decompose_marker(data, marker)
    model_data["week_centered"] = model_data["week"] - model_data["week"].mean()
    model = smf.mixedlm(
        "rgpts ~ marker_within + marker_between + C(week)",
        model_data,
        groups=model_data["participant_id"],
        re_formula="~week_centered",
    ).fit(reml=False, method="lbfgs")
    return model, model_data


def fit_omicron_model(data: pd.DataFrame, adjust: bool):
    model_data = data.dropna(
        subset=["rigidity", "rgpts", "omicron_concern"]
    ).copy()
    for variable in ["rigidity"] + (["omicron_concern"] if adjust else []):
        model_data[f"{variable}_z"] = zscore_sample(model_data[variable])
        model_data[f"{variable}_between"] = model_data.groupby("participant_id")[
            f"{variable}_z"
        ].transform("mean")
        model_data[f"{variable}_within"] = (
            model_data[f"{variable}_z"] - model_data[f"{variable}_between"]
        )
    terms = "rigidity_within + rigidity_between + C(week)"
    if adjust:
        terms += " + omicron_concern_within + omicron_concern_between"
    model = smf.mixedlm(
        f"rgpts ~ {terms}", model_data, groups=model_data["participant_id"]
    ).fit(reml=False)
    return model, model_data


def sensitivity_row(
    analysis: str, model, term: str, data: pd.DataFrame
) -> dict[str, float | int | str]:
    estimate = float(model.params[term])
    standard_error = float(model.bse[term])
    return {
        "Analysis": analysis,
        "b": estimate,
        "SE": standard_error,
        "CI low": estimate - 1.96 * standard_error,
        "CI high": estimate + 1.96 * standard_error,
        "P": float(model.pvalues[term]),
        "Person-weeks": len(data),
        "Participants": data["participant_id"].nunique(),
    }


random_slope_model, random_slope_data = fit_random_session_slope(panel, "rigidity")
exclude_week1_model, exclude_week1_data = fit_between_within(
    panel[panel["week"] > 1].copy(), "rigidity"
)
omicron_panel = panel.dropna(subset=["omicron_concern"]).copy()
assert len(omicron_panel) == 708
assert omicron_panel["participant_id"].nunique() == 131
omicron_unadjusted_model, omicron_unadjusted_data = fit_omicron_model(
    omicron_panel, adjust=False
)
omicron_adjusted_model, omicron_adjusted_data = fit_omicron_model(
    omicron_panel, adjust=True
)

sensitivity_results = pd.DataFrame(
    [
        sensitivity_row(
            "Primary: categorical week effects",
            rigidity_model,
            "marker_within",
            rigidity_model_data,
        ),
        sensitivity_row(
            "Participant-specific linear session slopes",
            random_slope_model,
            "marker_within",
            random_slope_data,
        ),
        sensitivity_row(
            "Exclude week 1",
            exclude_week1_model,
            "marker_within",
            exclude_week1_data,
        ),
        sensitivity_row(
            "Weeks 3–8",
            omicron_unadjusted_model,
            "rigidity_within",
            omicron_unadjusted_data,
        ),
        sensitivity_row(
            "Weeks 3–8 + Omicron concern",
            omicron_adjusted_model,
            "rigidity_within",
            omicron_adjusted_data,
        ),
    ]
)
omicron_prevalence = (
    omicron_panel.groupby("week")["omicron_concern"]
    .agg(proportion="mean", n="size")
    .reset_index()
)
omicron_prevalence["percent"] = 100 * omicron_prevalence["proportion"]

fig = plt.figure(figsize=(13.2, 4.7))
outer = fig.add_gridspec(1, 2, width_ratios=[0.88, 2.12], wspace=0.24)
axis_a = fig.add_subplot(outer[0])
right = outer[1].subgridspec(1, 3, width_ratios=[1.45, 1.65, 1.05], wspace=0.03)
axis_labels = fig.add_subplot(right[0])
axis_b = fig.add_subplot(right[1], sharey=axis_labels)
axis_values = fig.add_subplot(right[2], sharey=axis_labels)
fig.subplots_adjust(left=0.065, right=0.985, top=0.78, bottom=0.18)

axis_a.bar(
    omicron_prevalence["week"],
    omicron_prevalence["percent"],
    color=COLORS["blue"],
    width=0.66,
)
for row in omicron_prevalence.itertuples():
    axis_a.text(row.week, row.percent + 1.4, f"{row.percent:.0f}%", ha="center", fontsize=9.2)
axis_a.set(
    xlabel="Session week",
    ylabel="Participants concerned about Omicron (%)",
    xticks=omicron_prevalence["week"],
    ylim=(0, 70),
)
axis_a.set_title("a  Omicron concern during weeks 3–8", loc="left", fontweight="bold")

display_sensitivity = sensitivity_results.iloc[::-1].reset_index(drop=True)
y = np.arange(len(display_sensitivity))
for index, row in display_sensitivity.iterrows():
    primary = row["Analysis"].startswith("Primary")
    axis_b.errorbar(
        row["b"],
        index,
        xerr=1.96 * row["SE"],
        fmt="s",
        color=COLORS["trait"] if primary else "#5F7480",
        ecolor=COLORS["dark"],
        capsize=3.2,
        markersize=5.5,
    )
    axis_labels.text(0.98, index, row["Analysis"], ha="right", va="center", fontsize=9.1)
    p_value = "< 0.001" if row["P"] < 0.001 else f"= {row['P']:.3f}"
    axis_values.text(
        0.02,
        index,
        f"{row['b']:.2f} ({row['CI low']:.2f}, {row['CI high']:.2f})\nP {p_value}",
        ha="left",
        va="center",
        fontsize=8.5,
    )
axis_b.axvline(0, color="#7A7A7A", lw=1)
axis_b.set(
    yticks=[],
    xlabel="Within-person rigidity coefficient, b\n(R-GPTS points per 1 s.d. rigidity)",
    xlim=(-0.25, 0.95),
)
axis_b.grid(axis="x", color=COLORS["light"], lw=0.7, alpha=0.7)
for axis in [axis_labels, axis_values]:
    axis.set(xlim=(0, 1), xticks=[], yticks=[])
    for spine in axis.spines.values():
        spine.set_visible(False)
axis_values.set_title("Estimate (95% CI)", loc="left", fontsize=9.2, fontweight="bold")
fig.text(
    axis_labels.get_position().x0,
    0.815,
    "b  Sensitivity of the within-person rigidity estimate",
    ha="left",
    fontweight="bold",
    fontsize=11,
)
fig.suptitle(
    "Supplementary Fig. 2 | Sensitivity to repeated testing and Omicron",
    fontweight="bold",
)
fig.savefig(OUT / "supplementary_figure_2.png", bbox_inches="tight")
fig.savefig(OUT / "supplementary_figure_2.pdf", bbox_inches="tight")
plt.show()

display(
    sensitivity_results.style.format(
        {"b": "{:+.3f}", "SE": "{:.3f}", "CI low": "{:+.3f}", "CI high": "{:+.3f}", "P": "{:.4f}"}
    )
)
omicron_effects = pd.DataFrame(
    [
        coefficient_row(
            omicron_adjusted_model,
            "omicron_concern_between",
            "Omicron concern",
            "Between people",
        ),
        coefficient_row(
            omicron_adjusted_model,
            "omicron_concern_within",
            "Omicron concern",
            "Within person",
        ),
    ]
)
display(omicron_effects.style.format({"b": "{:+.3f}", "SE": "{:.3f}", "P": "{:.4f}"}))

omicron_prevalence.to_csv(OUT / "figure_s2a_omicron_prevalence.csv", index=False)
sensitivity_results.to_csv(OUT / "figure_s2b_rigidity_sensitivity.csv", index=False)

rigidity_unadjusted = float(omicron_unadjusted_model.params["rigidity_within"])
rigidity_adjusted = float(omicron_adjusted_model.params["rigidity_within"])
assert np.isclose(rigidity_unadjusted, 0.293, atol=0.02)
assert np.isclose(rigidity_adjusted, 0.296, atol=0.02)
assert abs(rigidity_adjusted - rigidity_unadjusted) < 0.02

C:\Users\ps967\AppData\Local\Temp\1\ipykernel_31596\1429653417.py:181: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Analysis,b,SE,CI low,CI high,P,Person-weeks,Participants
0,Primary: categorical week effects,+0.494,0.196,+0.110,+0.877,0.0117,1000,150
1,Participant-specific linear session slopes,+0.489,0.196,+0.105,+0.873,0.0126,1000,150
2,Exclude week 1,+0.262,0.200,-0.130,+0.655,0.1905,850,140
3,Weeks 3–8,+0.293,0.230,-0.157,+0.744,0.2015,708,131
4,Weeks 3–8 + Omicron concern,+0.296,0.230,-0.154,+0.747,0.1972,708,131


,Marker,Level,b,SE,CI low,CI high,z,P,Person-weeks,Participants
0,Omicron concern,Between people,+0.748,0.671,-0.566457,2.062330,1.115309,0.2647,708,131
1,Omicron concern,Within person,+0.149,0.356,-0.548824,0.847711,0.419481,0.6749,708,131


## Supplementary Figure 3: volatility priors distinguish people, not weeks

The HGF volatility prior is decomposed with the same between-person/within-person model used for Figure 4. It is included alongside win-switch rate and rigidity to show that volatility priors are higher in participants with greater average persecution but do not detectably track a participant's weekly deviations in persecution.

The volatility parameter was estimated before this public Python workflow. Its session-level fitted values are supplied in `weekly_measures.csv`; the analysis below is fully reproduced, while HGF estimation itself requires the original model-fitting environment.

In [11]:
volatility_model, volatility_model_data = fit_between_within(panel, "mu03")
volatility_coefficients = pd.DataFrame(
    [
        coefficient_row(
            volatility_model,
            "marker_between",
            "Volatility prior (μ₃⁰)",
            "Between people",
        ),
        coefficient_row(
            volatility_model,
            "marker_within",
            "Volatility prior (μ₃⁰)",
            "Within person",
        ),
    ]
)
supplementary_figure_3_coefficients = pd.concat(
    [figure4_coefficients, volatility_coefficients], ignore_index=True
)
supplementary_figure_3_contrasts = pd.concat(
    [
        figure4_contrasts,
        pd.DataFrame(
            [
                {
                    "Marker": "Volatility prior (μ₃⁰)",
                    **wald_difference(
                        volatility_model, "marker_between", "marker_within"
                    ),
                }
            ]
        ),
    ],
    ignore_index=True,
)

markers = [
    ("Win-switch rate", COLORS["trait"]),
    ("Win-switch rigidity", COLORS["state"]),
    ("Volatility prior (μ₃⁰)", COLORS["volatility"]),
]
levels = ["Between people", "Within person"]
fig, axis = plt.subplots(figsize=(8.7, 5.2))
x = np.arange(len(levels))
width = 0.23
for offset, (marker, color) in zip([-width, 0, width], markers):
    data = (
        supplementary_figure_3_coefficients.query("Marker == @marker")
        .set_index("Level")
        .reindex(levels)
    )
    bars = axis.bar(
        x + offset,
        data["b"],
        width,
        yerr=1.96 * data["SE"],
        capsize=4,
        color=color,
        label=marker,
        error_kw={"ecolor": COLORS["dark"], "lw": 1.3},
    )
    for bar, (_, row) in zip(bars, data.iterrows()):
        endpoint = row["b"] + np.sign(row["b"] or 1) * 1.96 * row["SE"]
        axis.text(
            bar.get_x() + bar.get_width() / 2,
            endpoint + (0.16 if row["b"] >= 0 else -0.25),
            "n.s." if row["P"] >= 0.05 else ("P < 0.001" if row["P"] < 0.001 else f"P = {row['P']:.3f}"),
            ha="center",
            va="bottom" if row["b"] >= 0 else "top",
            fontsize=8.5,
        )
axis.axhline(0, color="#999999", lw=1)
axis.set_xticks(
    x,
    [
        "Between people\n(who is more paranoid?)",
        "Within person\n(which weeks are more paranoid?)",
    ],
)
axis.set_ylabel(
    "Association with R-GPTS persecution\n"
    "(b per marker s.d., 95% CI; week-adjusted)"
)
axis.set_ylim(-4.6, 4.0)
axis.legend(frameon=False, loc="upper right", fontsize=9)
axis.set_title(
    "Supplementary Fig. 3 | Volatility priors distinguish people, not weeks",
    fontweight="bold",
)
fig.tight_layout()
fig.savefig(OUT / "supplementary_figure_3.png", bbox_inches="tight")
fig.savefig(OUT / "supplementary_figure_3.pdf", bbox_inches="tight")
plt.show()

display(
    supplementary_figure_3_coefficients.style.format(
        {"b": "{:+.3f}", "SE": "{:.3f}", "CI low": "{:+.3f}", "CI high": "{:+.3f}", "P": "{:.4f}"}
    )
)
display(
    supplementary_figure_3_contrasts.style.format(
        {"difference": "{:+.3f}", "SE": "{:.3f}", "CI low": "{:+.3f}", "CI high": "{:+.3f}", "z": "{:+.3f}", "P": "{:.4f}"}
    )
)
supplementary_figure_3_coefficients.to_csv(
    OUT / "figure_s3_coefficients.csv", index=False
)
supplementary_figure_3_contrasts.to_csv(
    OUT / "figure_s3_contrasts.csv", index=False
)

volatility_locked = volatility_coefficients.set_index("Level")
assert np.isclose(volatility_locked.loc["Between people", "b"], 2.113, atol=0.02)
assert np.isclose(volatility_locked.loc["Within person", "b"], 0.089, atol=0.02)
assert volatility_locked.loc["Between people", "P"] < 0.02
assert volatility_locked.loc["Within person", "P"] > 0.5

C:\Users\ps967\AppData\Local\Temp\1\ipykernel_31596\366410472.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Marker,Level,b,SE,CI low,CI high,z,P,Person-weeks,Participants
0,Win-switch rate,Between people,+1.742,0.548,+0.668,+2.816,3.180522,0.0015,1000,150
1,Win-switch rate,Within person,+0.090,0.220,-0.341,+0.522,0.410304,0.6816,1000,150
2,Win-switch rigidity,Between people,-2.226,0.673,-3.544,-0.907,-3.309266,0.0009,1000,150
3,Win-switch rigidity,Within person,+0.494,0.196,+0.110,+0.877,2.521227,0.0117,1000,150
4,Volatility prior (μ₃⁰),Between people,+2.113,0.837,+0.473,+3.754,2.525518,0.0116,1000,150
5,Volatility prior (μ₃⁰),Within person,+0.089,0.164,-0.232,+0.410,0.542086,0.5878,1000,150


,Marker,difference,SE,CI low,CI high,z,P
0,Win-switch rate,+1.652,0.588,+0.498,+2.805,+2.807,0.0050
1,Win-switch rigidity,-2.719,0.699,-4.089,-1.349,-3.891,0.0001
2,Volatility prior (μ₃⁰),+2.025,0.852,+0.355,+3.694,+2.376,0.0175


## Final validation, source-data exports and manuscript map

- **Main Fig. 2:** trust curves by volatility-prior tertile and person-mean win-switch rate by persecution threshold.
- **Main Fig. 3:** descriptive within-person ordering of persecution, win-switch rate and rigidity.
- **Main Fig. 4:** separate between-person and within-person mixed-model associations for win-switch rate and rigidity.
- **Supplementary Fig. 1:** rigidity split-half reliability, leave-one-facet-out checks and within-person permutation null.
- **Supplementary Fig. 2:** repeated-testing and Omicron sensitivity.
- **Supplementary Fig. 3:** between-person and within-person associations including the HGF volatility prior.
- **Supplementary Table 1:** aggregate sample characteristics.
- **Supplementary Table 2:** weekly retention and analysis-specific samples.

The final cell exports figure-level statistics and applies locked checks to every headline quantity. A failed assertion indicates that the data, preprocessing or model specification no longer reproduces the audited submission result.

In [12]:
# Main-figure numerical source data.
figure2_curve.to_csv(OUT / "figure2a_trust_curve.csv", index=False)
figure2a_tests.to_csv(OUT / "figure2a_tests.csv", index=False)
figure2_group_summary.assign(
    mann_whitney_U=figure2b_test.statistic,
    mann_whitney_P=figure2b_test.pvalue,
    spearman_rho=figure2b_rho.statistic,
    spearman_P=figure2b_rho.pvalue,
).to_csv(OUT / "figure2b_statistics.csv", index=False)
ranked_summary.to_csv(OUT / "figure3_ranked_week_summary.csv", index=False)
figure3_tests.to_csv(OUT / "figure3_paired_tests.csv", index=False)
figure4_coefficients.to_csv(OUT / "figure4_coefficients.csv", index=False)
figure4_contrasts.to_csv(OUT / "figure4_wald_contrasts.csv", index=False)
reliability_results.to_csv(OUT / "supplementary_reliability_summary.csv", index=False)
omicron_effects.to_csv(OUT / "supplementary_omicron_effects.csv", index=False)

# Analysis-ready derived panel (contains study pseudonyms only).
panel.to_csv(OUT / "derived_person_week_panel.csv", index=False)
software_versions = pd.DataFrame(
    {
        "Package": ["Python", "pandas", "NumPy", "SciPy", "statsmodels"],
        "Version": [
            platform.python_version(),
            pd.__version__,
            np.__version__,
            scipy.__version__,
            statsmodels.__version__,
        ],
    }
)
software_versions.to_csv(OUT / "software_versions.csv", index=False)

checks = [
    ("Primary panel: 150 participants", panel["participant_id"].nunique() == 150),
    ("Primary panel: 1,000 person-weeks", len(panel) == 1_000),
    ("Figure 2: high-persecution group n=23", len(high_person) == 23),
    ("Figure 2: group comparison P<0.001", figure2b_test.pvalue < 0.001),
    ("Figure 3: ranked-week sample n=68", ranked["participant_id"].nunique() == 68),
    ("Figure 3: rigidity P≈0.012", np.isclose(figure3_rigidity["p"], 0.0115, atol=0.001)),
    ("Figure 3: win-switch rate remains null", figure3_wsr["p"] > 0.4),
    ("Figure 4: WSR between b≈1.74", np.isclose(locked.loc[("Win-switch rate", "Between people"), "b"], 1.742, atol=0.01)),
    ("Figure 4: WSR within is null", locked.loc[("Win-switch rate", "Within person"), "P"] > 0.6),
    ("Figure 4: rigidity between b≈−2.23", np.isclose(locked.loc[("Win-switch rigidity", "Between people"), "b"], -2.226, atol=0.02)),
    ("Figure 4: rigidity within b≈0.49", np.isclose(locked.loc[("Win-switch rigidity", "Within person"), "b"], 0.494, atol=0.01)),
    ("Supplementary Fig. 1: rigidity rSB rounds to 0.83", round(rigidity_sb, 2) == 0.83),
    ("Supplementary Fig. 1: leave-one-out P range 0.003–0.038", round(leave_one_out["P"].min(), 3) == 0.003 and round(leave_one_out["P"].max(), 3) == 0.038),
    ("Supplementary Fig. 1: permutation P rounds to 0.011", round(permutation_result["P"], 3) == 0.011),
    ("Supplementary Fig. 2: Omicron adjustment Δb≈0", abs(rigidity_adjusted - rigidity_unadjusted) < 0.02),
    ("Supplementary Fig. 3: volatility between P≈0.012", np.isclose(volatility_locked.loc["Between people", "P"], 0.012, atol=0.002)),
    ("Supplementary Fig. 3: volatility within is null", volatility_locked.loc["Within person", "P"] > 0.5),
]
validation = pd.DataFrame(checks, columns=["Check", "Passed"])
display(validation)
assert validation["Passed"].all(), "At least one locked manuscript check failed."

expected_artifacts = [
    "figure2_main.png",
    "figure2_main.pdf",
    "figure3_main.png",
    "figure3_main.pdf",
    "figure4_main.png",
    "figure4_main.pdf",
    "supplementary_figure_1.png",
    "supplementary_figure_1.pdf",
    "supplementary_figure_2.png",
    "supplementary_figure_2.pdf",
    "supplementary_figure_3.png",
    "supplementary_figure_3.pdf",
    "supplementary_table_1_sample_characteristics.csv",
    "supplementary_table_2_weekly_retention.csv",
    "supplementary_table_2_analysis_samples.csv",
]
missing_artifacts = [name for name in expected_artifacts if not (OUT / name).exists()]
assert not missing_artifacts, f"Missing expected outputs: {missing_artifacts}"
display(pd.DataFrame({"Expected artifact": expected_artifacts}))
print(f"All {len(checks)} locked checks passed. Reproduced outputs: {OUT}")

,Check,Passed
0,Primary panel: 150 participants,True
1,"Primary panel: 1,000 person-weeks",True
2,Figure 2: high-persecution group n=23,True
3,Figure 2: group comparison P<0.001,True
4,Figure 3: ranked-week sample n=68,True
5,Figure 3: rigidity P≈0.012,True
6,Figure 3: win-switch rate remains null,True
7,Figure 4: WSR between b≈1.74,True
8,Figure 4: WSR within is null,True
9,Figure 4: rigidity between b≈−2.23,True


,Expected artifact
0,figure2_main.png
1,figure2_main.pdf
2,figure3_main.png
3,figure3_main.pdf
4,figure4_main.png
5,figure4_main.pdf
6,supplementary_figure_1.png
7,supplementary_figure_1.pdf
8,supplementary_figure_2.png
9,supplementary_figure_2.pdf


All 17 locked checks passed. Reproduced outputs: C:\Users\ps967\Desktop\PhD\longitudinal\prl-longitudinal\outputs
